<a href="https://colab.research.google.com/github/Abdo-Kahrawi/StudyPlanner/blob/branch_abdo/make_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [144]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv
#This file is going to be dedicated to working with preprocessing and understanding the data

In [9]:
# Store them in a dictionary
dfs = {
    'Assessments': pd.read_csv('assessments.csv'),
    'Courses': pd.read_csv('courses.csv'),
    'Student Assessment': pd.read_csv('studentAssessment.csv'),
    'Student Registration': pd.read_csv('studentRegistration.csv'),
    'VLE': pd.read_csv('vle.csv'),
    'Student VLE': pd.read_csv('studentVle.csv', quoting=csv.QUOTE_NONE) #Due to a parsing error at the line 1133148
}

# Loop through and inspect each one
for name, df in dfs.items():
    print(f"=== {name} ===")
    print(df.info())
    print(df.head(3))
    print("\n" + "="*30 + "\n")

#There is a problem with the date column in assessments. (Nulls)
#There is a problem with the score column in student assessments (Nulls)
#There is a problem with the date_registration column in student registrations
#There is a problem with the date_unregistration column in student registrations (Invalid data + nulls)
#There is a problem with the week_from and week_to columns in VLE (Invalid data)
#There is a problem with the student VLE csv (contains duplicates)




=== Assessments ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    object 
 1   code_presentation  206 non-null    object 
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    object 
 4   date               195 non-null    float64
 5   weight             206 non-null    float64
dtypes: float64(2), int64(1), object(3)
memory usage: 9.8+ KB
None
  code_module code_presentation  id_assessment assessment_type   date  weight
0         AAA             2013J           1752             TMA   19.0    10.0
1         AAA             2013J           1753             TMA   54.0    20.0
2         AAA             2013J           1754             TMA  117.0    20.0


=== Courses ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 col

In [26]:
#There is a problem with the date column in assessments. (Nulls)
#Let me explain: the value in the date column represents the official deadline after n days of the start of the course.
dfs['Assessments'].info()
dfs['Assessments'].isnull().sum() #It is a given that a mid or a final exam don't have deadline submission after n days of the start of the course.
# I have an idea of separeting unrelated columns and grouping only related or dependent ones together, but this is a lot.
# I could add a label next to a NaN value that says: "Exam" instead of a regular assessment with a deadline.
dfs['Assessments'].duplicated().sum() #No duplicates.

#An idea to mitigate this null problem in the date column is to write the length of the course itself as a default, using course_length from the courses.csv
dfs['Assessments'].head()
dfs['Assessments']['assessment_type'].unique()
#If assessment_type = Exam, replace the nulls with the module_length.
#dfs['Assessments']['date'] = np.where((dfs['Assessments']['assessment_type']=='Exam')&(dfs['Assessments']['code_module']==dfs['Courses']['code_module'])
                                      # & (dfs['Assessments']['code_presentation']==dfs['Courses']['code_presentation']), dfs['Courses']['module_presentation_length'])
# 1. Merge module_presentation_length into Assessments based on matching keys
df_ass = dfs['Assessments'].merge(
    dfs['Courses'][['code_module', 'code_presentation', 'module_presentation_length']],
    on=['code_module', 'code_presentation'],
    how='left'
)

# 2. Use np.where to fill missing dates for Exams, keeping existing dates for everything else
df_ass['date'] = np.where(
    (df_ass['assessment_type'] == 'Exam') & (df_ass['date'].isna()),
    df_ass['module_presentation_length'],
    df_ass['date']
)

# 3. Clean up the temporary column if needed
df_ass = df_ass.drop(columns=['module_presentation_length'])

# Update your dictionary
dfs['Assessments'] = df_ass

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    object 
 1   code_presentation  206 non-null    object 
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    object 
 4   date               195 non-null    float64
 5   weight             206 non-null    float64
dtypes: float64(2), int64(1), object(3)
memory usage: 9.8+ KB


In [30]:

dfs['Assessments'].info()
dfs['Assessments'].isnull().sum()
dfs['Assessments'][dfs['Assessments']['assessment_type']=='Exam'].head()

#We solved the first problem: Null Date of Assessments


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    object 
 1   code_presentation  206 non-null    object 
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    object 
 4   date               206 non-null    float64
 5   weight             206 non-null    float64
dtypes: float64(2), int64(1), object(3)
memory usage: 9.8+ KB


,code_module,code_presentation,id_assessment,assessment_type,date,weight
5,AAA,2013J,1757,Exam,268.0,100.0
11,AAA,2014J,1763,Exam,269.0,100.0
23,BBB,2013B,14990,Exam,240.0,100.0
35,BBB,2013J,15002,Exam,268.0,100.0
47,BBB,2014B,15014,Exam,234.0,100.0


In [21]:
dfs['Courses'].info()
dfs['Courses'].head()
dfs['Courses'].isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   code_module                 22 non-null     object
 1   code_presentation           22 non-null     object
 2   module_presentation_length  22 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 660.0+ bytes


,0
code_module,0
code_presentation,0
module_presentation_length,0


In [32]:
#There is a problem with the score column in student assessments (Nulls)
dfs['Student Assessment'].info()
dfs['Student Assessment'].isnull().sum()
#dfs['Student Assessment'].head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173912 entries, 0 to 173911
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id_assessment   173912 non-null  int64  
 1   id_student      173912 non-null  int64  
 2   date_submitted  173912 non-null  int64  
 3   is_banked       173912 non-null  int64  
 4   score           173739 non-null  float64
dtypes: float64(1), int64(4)
memory usage: 6.6 MB


,0
id_assessment,0
id_student,0
date_submitted,0
is_banked,0
score,173


In [33]:
df_score_null = dfs['Student Assessment'][dfs['Student Assessment']['score'].isna()]
df_score_null.head()
#The quest now is about the late submisions and what can possibly result in NaN scores, given that the students have submitted the assessments.
#Merge the assessment and student assessment tables. Ask about what is_banked is for.

,id_assessment,id_student,date_submitted,is_banked,score
215,1752,721259,22,0,NaN
937,1754,260355,127,0,NaN
2364,1760,2606802,180,0,NaN
3358,14984,186780,77,0,NaN
3914,14984,531205,26,0,NaN


In [73]:
# 1. Merge module_presentation_length into Assessments based on matching keys
df_ass1 = dfs['Assessments'].merge(
    dfs['Student Assessment'],
    on=['id_assessment'],
    how='left'
)

# 2. Use np.where to fill missing dates for Exams, keeping existing dates for everything else
df_ass1['Difference'] = df_ass1['date_submitted'] - df_ass1['date']
df_ass1['Submission Category'] = np.where(df_ass1['Difference']< 0, 'Early', 'Late')

df_ass1.head()
df_ass1[df_ass1['score'].isna()].head()
df_ass1.isnull().sum()
#I am observing some interesting pattern in the data regarding the null thing.
df_ass1[(df_ass1['id_student'].isna())&(df_ass1['is_banked'].isna())&(df_ass1['date_submitted'].isna())].head(18)
#Good, know we know that all missing student ids match the exact same records for NaNs in is_banked and date_submitted and have NaN score.
#Should we drop 'em?

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score,Difference,Submission Category
1633,AAA,2013J,1757,Exam,268.0,100.0,NaN,NaN,NaN,NaN,NaN,Late
3150,AAA,2014J,1763,Exam,269.0,100.0,NaN,NaN,NaN,NaN,NaN,Late
14407,BBB,2013B,14990,Exam,240.0,100.0,NaN,NaN,NaN,NaN,NaN,Late
28783,BBB,2013J,15002,Exam,268.0,100.0,NaN,NaN,NaN,NaN,NaN,Late
38777,BBB,2014B,15014,Exam,234.0,100.0,NaN,NaN,NaN,NaN,NaN,Late
46186,BBB,2014J,15025,Exam,262.0,100.0,NaN,NaN,NaN,NaN,NaN,Late
53676,CCC,2014B,40087,Exam,241.0,100.0,NaN,NaN,NaN,NaN,NaN,Late
65128,CCC,2014J,40088,Exam,269.0,100.0,NaN,NaN,NaN,NaN,NaN,Late
98877,EEE,2013J,30713,Exam,235.0,100.0,NaN,NaN,NaN,NaN,NaN,Late
100658,EEE,2014B,30718,Exam,228.0,100.0,NaN,NaN,NaN,NaN,NaN,Late


In [74]:

df_ass1 = df_ass1.dropna(subset=['id_student'])
df_ass1.isnull().sum()
#We still have 173 records with NaN scores. What could possibly have gone wrong with those assessments, so they didn't get graded?
df_null_score = df_ass1[df_ass1['score'].isna()]
df_null_score.info()
#Now, we try to get the assessment_type of these.
df_null_score['assessment_type'].unique()
#We now know that all of these null scores are TMA assessment type. Interesting.
df_null_score.head()
#Now, we try to get the submission category of these guys.

df_null_score['is_banked'].value_counts()
df_null_score['code_module'].value_counts()
#Also, 20 of them were early submissions. It can't be because of late submissions for all.
df_null_score['Submission Category'].value_counts()
#We could check against the registration records of these courses.

<class 'pandas.core.frame.DataFrame'>
Index: 173 entries, 215 to 172867
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          173 non-null    object 
 1   code_presentation    173 non-null    object 
 2   id_assessment        173 non-null    int64  
 3   assessment_type      173 non-null    object 
 4   date                 173 non-null    float64
 5   weight               173 non-null    float64
 6   id_student           173 non-null    float64
 7   date_submitted       173 non-null    float64
 8   is_banked            173 non-null    float64
 9   score                0 non-null      float64
 10  Difference           173 non-null    float64
 11  Submission Category  173 non-null    object 
dtypes: float64(7), int64(1), object(4)
memory usage: 17.6+ KB


,count
Submission Category,
Late,153
Early,20


In [96]:
dfs['Student Registration'] = pd.read_csv('studentRegistration.csv')
df_studreg = dfs['Student Registration']
df_studreg.head()
#missing_score_students = df_studass_clean[df_studass_clean['score'].isna()]['id_student']
null_score_students = df_null_score['id_student']
df_studreg['Category'] = np.where(df_studreg['date_unregistration'].isna(), "Not Withdrawn", 'Withdrawn')
# Check their registration/withdrawal details
df_checking = df_studreg[df_studreg['id_student'].isin(null_score_students)]
df_checking.info()


<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 108 to 32167
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          210 non-null    object 
 1   code_presentation    210 non-null    object 
 2   id_student           210 non-null    int64  
 3   date_registration    210 non-null    float64
 4   date_unregistration  94 non-null     float64
 5   Category             210 non-null    object 
dtypes: float64(2), int64(1), object(3)
memory usage: 11.5+ KB


In [97]:
df_studreg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          32593 non-null  object 
 1   code_presentation    32593 non-null  object 
 2   id_student           32593 non-null  int64  
 3   date_registration    32548 non-null  float64
 4   date_unregistration  10072 non-null  float64
 5   Category             32593 non-null  object 
dtypes: float64(2), int64(1), object(3)
memory usage: 1.5+ MB


In [60]:
df_checking.head()
reg_studs_null_score = df_checking[df_checking['Category'] == "Not Withdrawn"]
reg_studs_null_score.info()
reg_studs_null_score.head()



<class 'pandas.core.frame.DataFrame'>
Index: 116 entries, 733 to 31682
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          116 non-null    object 
 1   code_presentation    116 non-null    object 
 2   id_student           116 non-null    int64  
 3   date_registration    116 non-null    float64
 4   date_unregistration  0 non-null      float64
 5   Category             116 non-null    object 
dtypes: float64(2), int64(1), object(3)
memory usage: 6.3+ KB


,code_module,code_presentation,id_student,date_registration,date_unregistration,Category
733,AAA,2014J,2606802,-37.0,NaN,Not Withdrawn
1256,BBB,2013B,484039,-53.0,NaN,Not Withdrawn
1361,BBB,2013B,502717,-21.0,NaN,Not Withdrawn
1423,BBB,2013B,511527,-95.0,NaN,Not Withdrawn
1458,BBB,2013B,515497,-24.0,NaN,Not Withdrawn


In [75]:
#Checking against the late category to determine what to do with this missing score.
reg_ids_null_score = reg_studs_null_score['id_student']
reg_ids_null_score.info()
df_score_dec = df_ass1[df_ass1['id_student'].isin(reg_ids_null_score) & df_ass1['score'].isna()]
df_score_dec.info()
df_score_dec['Submission Category'].value_counts()

<class 'pandas.core.series.Series'>
Index: 116 entries, 733 to 31682
Series name: id_student
Non-Null Count  Dtype
--------------  -----
116 non-null    int64
dtypes: int64(1)
memory usage: 5.9 KB
<class 'pandas.core.frame.DataFrame'>
Index: 111 entries, 2365 to 169469
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          111 non-null    object 
 1   code_presentation    111 non-null    object 
 2   id_assessment        111 non-null    int64  
 3   assessment_type      111 non-null    object 
 4   date                 111 non-null    float64
 5   weight               111 non-null    float64
 6   id_student           111 non-null    float64
 7   date_submitted       111 non-null    float64
 8   is_banked            111 non-null    float64
 9   score                0 non-null      float64
 10  Difference           111 non-null    float64
 11  Submission Category  111 non-null    objec

,count
Submission Category,
Late,104
Early,7


In [76]:
df_ass1['score'] = df_ass1['score'].fillna(0) #After some checking, we simply put a zero there.
df_ass1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 173912 entries, 0 to 173928
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   code_module          173912 non-null  object 
 1   code_presentation    173912 non-null  object 
 2   id_assessment        173912 non-null  int64  
 3   assessment_type      173912 non-null  object 
 4   date                 173912 non-null  float64
 5   weight               173912 non-null  float64
 6   id_student           173912 non-null  float64
 7   date_submitted       173912 non-null  float64
 8   is_banked            173912 non-null  float64
 9   score                173912 non-null  float64
 10  Difference           173912 non-null  float64
 11  Submission Category  173912 non-null  object 
dtypes: float64(7), int64(1), object(4)
memory usage: 17.2+ MB


In [79]:

ass_columns = ['id_assessment', 'id_student', 'date_submitted', 'Difference', 'Submission Category', 'is_banked', 'score']
dfs['Student Assessment'] = df_ass1[ass_columns]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173912 entries, 0 to 173911
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id_assessment   173912 non-null  int64  
 1   id_student      173912 non-null  int64  
 2   date_submitted  173912 non-null  int64  
 3   is_banked       173912 non-null  int64  
 4   score           173739 non-null  float64
dtypes: float64(1), int64(4)
memory usage: 6.6 MB


In [80]:
dfs['Student Assessment'].info()
#We finished the second problem in the data.

<class 'pandas.core.frame.DataFrame'>
Index: 173912 entries, 0 to 173928
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id_assessment        173912 non-null  int64  
 1   id_student           173912 non-null  float64
 2   date_submitted       173912 non-null  float64
 3   Difference           173912 non-null  float64
 4   Submission Category  173912 non-null  object 
 5   is_banked            173912 non-null  float64
 6   score                173912 non-null  float64
dtypes: float64(5), int64(1), object(1)
memory usage: 10.6+ MB


In [ ]:

#There is a problem with the week_from and week_to columns in VLE (Invalid data)
#There is a problem with the student VLE csv

In [119]:
#There is a problem with the date_registration column in student registrations
#We continue with df_studreg


df_studreg[df_studreg['date_registration'].isna()].head()
null_reg_studs = df_studreg[(df_studreg['date_registration'].isna())&(df_studreg['date_unregistration'])]['id_student'] #39 with a NaN registration but unregistered afterwards (Their registeration date has to be 0)
null_reg_studs.info()
fk_stds = df_studreg[(df_studreg['date_registration'].isna())&(df_studreg['date_unregistration'].isna())]['id_student']#6 With two Nans. They have to be omitted.

df_studreg_clean = df_studreg[~df_studreg['id_student'].isin(fk_stds)]
#df_studreg_clean[df_studreg_clean['id_student'].isin(null_reg_studs)]['date_registration'].fillna(0)
df_studreg_clean['date_registration'] = df_studreg_clean['date_registration'].fillna(0)

#Also, there is another point of view: if a student withdrew from the course and has a zero score in the assessments, should we really consider that
#row to be valuable in our assessment of student performance? Actually, we could do something. Gather all withdrawn students and exclude their marks if it's zero.
failed_stds = df_studreg[df_studreg['Category'] == 'Withdrawn']['id_student']

<class 'pandas.core.series.Series'>
Index: 39 entries, 2538 to 30058
Series name: id_student
Non-Null Count  Dtype
--------------  -----
39 non-null     int64
dtypes: int64(1)
memory usage: 624.0 bytes


/tmp/ipykernel_793/2223565725.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_studreg_clean['date_registration'] = df_studreg_clean['date_registration'].fillna(0)


In [108]:
df_studreg_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32587 entries, 0 to 32592
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          32587 non-null  object 
 1   code_presentation    32587 non-null  object 
 2   id_student           32587 non-null  int64  
 3   date_registration    32587 non-null  float64
 4   date_unregistration  10072 non-null  float64
 5   Category             32587 non-null  object 
dtypes: float64(2), int64(1), object(3)
memory usage: 1.7+ MB


In [120]:
df_ass2 = dfs['Student Assessment']
#Let me get clear with myself: I want to drop the guys that got 0 scores and withdrew. There is a way to get this:
#First get the ids of students that withdrew: already with us: failed_stds

df_ass2.info()
remove_mask = df_ass2['id_student'].isin(failed_stds)&(df_ass2['score'] == 0)
#Second, filter the assessments based upon this
df_ass_clean = df_ass2[~remove_mask]

<class 'pandas.core.frame.DataFrame'>
Index: 173912 entries, 0 to 173928
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id_assessment        173912 non-null  int64  
 1   id_student           173912 non-null  float64
 2   date_submitted       173912 non-null  float64
 3   Difference           173912 non-null  float64
 4   Submission Category  173912 non-null  object 
 5   is_banked            173912 non-null  float64
 6   score                173912 non-null  float64
dtypes: float64(5), int64(1), object(1)
memory usage: 10.6+ MB


In [123]:
df_ass_clean.info()
df_ass_clean.head() #This proved to be usesless because the

<class 'pandas.core.frame.DataFrame'>
Index: 173732 entries, 0 to 173928
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id_assessment        173732 non-null  int64  
 1   id_student           173732 non-null  float64
 2   date_submitted       173732 non-null  float64
 3   Difference           173732 non-null  float64
 4   Submission Category  173732 non-null  object 
 5   is_banked            173732 non-null  float64
 6   score                173732 non-null  float64
dtypes: float64(5), int64(1), object(1)
memory usage: 10.6+ MB


,id_assessment,id_student,date_submitted,Difference,Submission Category,is_banked,score
0,1752,11391.0,18.0,-1.0,Early,0.0,78.0
1,1752,28400.0,22.0,3.0,Late,0.0,70.0
2,1752,31604.0,17.0,-2.0,Early,0.0,72.0
3,1752,32885.0,26.0,7.0,Late,0.0,69.0
4,1752,38053.0,19.0,0.0,Late,0.0,79.0


In [125]:
# 1. Calculate the mean score for each assignment and add it as a column
df_ass_clean['assessment_mean'] = df_ass_clean.groupby('id_assessment')['score'].transform('mean')

# 2. Define a function or rules for the performance labels
def label_performance(row):
    score = row['score']
    avg = row['assessment_mean']

    if score < 60:
        return 'Terrible'
    elif score < (avg - 10):   # Way below average
        return 'Low'
    elif score <= (avg + 10):  # Close to average
        return 'Acceptable'
    else:                      # Way above average
        return 'Good'

# 3. Apply the function row-by-row
df_ass_clean['performance_label'] = df_ass_clean.apply(label_performance, axis=1)


/tmp/ipykernel_793/2815125140.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ass_clean['assessment_mean'] = df_ass_clean.groupby('id_assessment')['score'].transform('mean')
/tmp/ipykernel_793/2815125140.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ass_clean['performance_label'] = df_ass_clean.apply(label_performance, axis=1)


In [129]:
df_ass_clean.info()
df_ass_clean.head()
df_ass_clean[df_ass_clean['performance_label']=='Low']

dfs['Student Assessment'] = df_ass_clean

<class 'pandas.core.frame.DataFrame'>
Index: 173732 entries, 0 to 173928
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id_assessment        173732 non-null  int64  
 1   id_student           173732 non-null  float64
 2   date_submitted       173732 non-null  float64
 3   Difference           173732 non-null  float64
 4   Submission Category  173732 non-null  object 
 5   is_banked            173732 non-null  float64
 6   score                173732 non-null  float64
 7   assessment_mean      173732 non-null  float64
 8   performance_label    173732 non-null  object 
dtypes: float64(6), int64(1), object(2)
memory usage: 13.3+ MB


In [136]:
#There is a problem with the week_from and week_to columns in VLE (Invalid data)
#There is a problem with the student VLE csv
df_vle = dfs['VLE']
df_vle.info()
df_vle.head()
df_vle.isnull().sum()
#A nice idea is to remove all rows that have Nans in their weekfrom and to (Doesn't sound that good, though.)

active_types = ['oucontent', 'quiz', 'resource', 'url', 'externalquiz', 'glossary', 'dataplus']

# 2. Filter vle.csv to keep only active study content
vle_active = df_vle[df_vle['activity_type'].isin(active_types)].copy()

# 3. Merge with courses.csv to get the module presentation length (in days)
vle_active = vle_active.merge(
    dfs['Courses'][['code_module', 'code_presentation', 'module_presentation_length']],
    on=['code_module', 'code_presentation'],
    how='left'
)

# 4. Calculate total course weeks (rounding up using np.ceil)
vle_active['max_weeks'] = np.ceil(vle_active['module_presentation_length'] / 7)

# 5. Impute week_from with Week 1 if NaN
vle_active['week_from'] = vle_active['week_from'].fillna(1)

# 6. Impute week_to with max_weeks if NaN
vle_active['week_to'] = vle_active['week_to'].fillna(vle_active['max_weeks'])

# Clean up helper column if desired
vle_active.drop(columns=['max_weeks'], inplace=True)

vle_active.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6364 entries, 0 to 6363
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_site            6364 non-null   int64  
 1   code_module        6364 non-null   object 
 2   code_presentation  6364 non-null   object 
 3   activity_type      6364 non-null   object 
 4   week_from          1121 non-null   float64
 5   week_to            1121 non-null   float64
dtypes: float64(2), int64(1), object(3)
memory usage: 298.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4744 entries, 0 to 4743
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id_site                     4744 non-null   int64  
 1   code_module                 4744 non-null   object 
 2   code_presentation           4744 non-null   object 
 3   activity_type               4744 non-null   object

In [138]:
vle_active.head()
vle_active = vle_active.drop('module_presentation_length', axis=1)
dfs['VLE']=vle_active

In [151]:


# Solution: Ignore quote parsing entirely while reading
df_studvle = pd.read_csv('studentVle.csv', quoting=csv.QUOTE_NONE)
# Clean a specific column
#df_studvle['code_module'] = df_studvle['code_module'].astype(str).str.replace('"', '').str.strip()

# Clean all object/string columns at once
for col in df_studvle.select_dtypes(include='object').columns:
    df_studvle[col] = df_studvle[col].astype(str).str.replace('"', '').str.strip()

df_studvle.columns = df_studvle.columns.str.replace('"', '').str.strip()

df_studvle.info()
df_studvle.head()

df_studvle.tail()
#First Question: what to do when all of the values in the whole csv are in double quotes? how to remove those double quotes?
#Second Question: date here is in negative. Does it simply mean that the student "did" the VLE before the deadline?
#Third Question: What about sum click? What does it mean? The sum of clicks by the student on the VLE?

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1133148 entries, 0 to 1133147
Data columns (total 6 columns):
 #   Column             Non-Null Count    Dtype 
---  ------             --------------    ----- 
 0   code_module        1133148 non-null  object
 1   code_presentation  1133148 non-null  object
 2   id_student         1133148 non-null  object
 3   id_site            1133148 non-null  object
 4   date               1133148 non-null  object
 5   sum_click          1133148 non-null  object
dtypes: object(6)
memory usage: 51.9+ MB


,code_module,code_presentation,id_student,id_site,date,sum_click
1133143,BBB,2013J,492726,703898,199,1
1133144,BBB,2013J,492726,703898,199,1
1133145,BBB,2013J,492726,703898,199,7
1133146,BBB,2013J,492726,703721,199,3
1133147,BBB,2013J,492726,nan,nan,nan


In [156]:
df_studvle.isnull().sum()
df_studvle[df_studvle['id_site']=='nan'].info()
df_studvle = df_studvle[df_studvle['id_site'] != 'nan']
df_studvle.tail()

dfs['Student VLE'] = df_studvle

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   code_module        0 non-null      object
 1   code_presentation  0 non-null      object
 2   id_student         0 non-null      object
 3   id_site            0 non-null      object
 4   date               0 non-null      object
 5   sum_click          0 non-null      object
dtypes: object(6)
memory usage: 0.0+ bytes
